# Face Follower — Client V3 (MediaPipe)

V3 is focused specifically on the two physical behaviors observed on the robot:

1. **Weak FORWARD:** keep FORWARD as a continuous full-power command on the ESP32/L298N instead of pulsing it.
2. **RIGHT overshoot:** use a gentler RIGHT motor pattern in the `.ino` plus a shorter RIGHT correction pulse in Python.

The existing ESP32 endpoints are preserved:
`/motor?dir=forward|backward|left|right|stop`

**Wiring stays unchanged for this V3:** IN1=13, IN2=12, IN3=14, IN4=15, while ENA and ENB remain tied to 5V.

Before running, set `ESP32_IP` to the address printed by Serial Monitor.


In [1]:
# ==================== INSTALL / IMPORT ====================
# Run this cell once in a fresh environment.
# After pip finishes, RESTART the Jupyter kernel, then run the import cell again.

%pip install "mediapipe==0.10.21" "numpy<2" "opencv-contrib-python<5" requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\erinx\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import time
import threading
mp_face_detection = mp.solutions.face_detection

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)


OpenCV: 4.11.0
MediaPipe: 0.10.21
NumPy: 1.26.4


In [3]:
# ==================== V3 CONFIG ====================
ESP32_IP = "10.158.116.178"
STREAM_URL = f"http://{ESP32_IP}:81/stream"
MOTOR_URL = f"http://{ESP32_IP}/motor"

# Position control
CENTER_DEADZONE = 0.18
CLOSE_FACE_RATIO = 0.35

# MediaPipe
MIN_DETECTION_CONFIDENCE = 0.55
MODEL_SELECTION = 0
MAX_NUM_FACES = 5

# Smoothing / stability
FACE_SMOOTHING_ALPHA = 0.30
DIRECTION_CONFIRM_FRAMES = 3
NO_FACE_STOP_FRAMES = 5

# HTTP
MOTOR_REQUEST_TIMEOUT = 3.0
SAME_COMMAND_HEARTBEAT = 1.5

# V3 turn pulse tuning.
# LEFT was already acceptable, so it gets a moderate pulse.
# RIGHT was overshooting, so it gets a much shorter pulse.
LEFT_TURN_PULSE_MS = 120
RIGHT_TURN_PULSE_MS = 65

# Minimum gap before another turn correction is allowed.
TURN_COOLDOWN_MS = 90

WINDOW_NAME = "Face Follower V3 - Weak Forward + Right Overshoot"
mp_face_detection = mp.solutions.face_detection


### V3 control changes

- FORWARD remains continuous so the robot receives the maximum available drive from the current L298N wiring.
- RIGHT on the ESP32 is now a gentler one-wheel turn instead of a full pivot.
- Python additionally limits RIGHT to a short **65 ms** correction pulse.
- LEFT starts at **120 ms** because the current LEFT response was acceptable.
- Deadzone increased to **18%** and command confirmation to **3 frames** to reduce unnecessary corrections.
- ENA/ENB remain tied to 5V; this version does **not** require PWM rewiring.


In [4]:
# ==================== MEDIAPIPE DETECTOR ====================
# MediaPipe expects RGB input. It returns normalized bounding-box coordinates.

face_detector = mp_face_detection.FaceDetection(
    model_selection=MODEL_SELECTION,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
)

print("MediaPipe Face Detection ready.")


MediaPipe Face Detection ready.


In [5]:
# ==================== V3 MOTOR COMMAND SENDER ====================
# FORWARD and STOP are state commands.
# LEFT and RIGHT are correction pulses. After each turn pulse the robot stops,
# waits for a new camera decision, then continues forward if the face is centered.

_last_sent_dir = None
_last_sent_time = 0.0
_motor_request_in_flight = False
_last_turn_time = 0.0
_command_lock = threading.Lock()

def _http_motor(direction):
    try:
        response = requests.get(
            MOTOR_URL,
            params={"dir": direction},
            timeout=MOTOR_REQUEST_TIMEOUT,
        )
        if response.status_code == 200:
            print(f"[motor] {direction} -> 200")
            return True
        print(f"[motor] {direction} -> HTTP {response.status_code}")
    except requests.RequestException as e:
        print(f"[motor] {direction} failed: {e}")
    return False

def send_direction(direction: str):
    global _last_sent_dir, _last_sent_time
    global _motor_request_in_flight, _last_turn_time

    now = time.time()

    # LEFT/RIGHT = short correction pulses.
    if direction in ("left", "right"):
        with _command_lock:
            if _motor_request_in_flight:
                return

            if (now - _last_turn_time) * 1000.0 < TURN_COOLDOWN_MS:
                return

            _motor_request_in_flight = True
            _last_turn_time = now
            _last_sent_dir = direction
            _last_sent_time = now

        pulse_ms = LEFT_TURN_PULSE_MS if direction == "left" else RIGHT_TURN_PULSE_MS

        def turn_worker():
            global _motor_request_in_flight, _last_sent_dir, _last_sent_time
            try:
                _http_motor(direction)
                time.sleep(pulse_ms / 1000.0)
                _http_motor("stop")
                with _command_lock:
                    _last_sent_dir = "stop"
                    _last_sent_time = time.time()
            finally:
                with _command_lock:
                    _motor_request_in_flight = False

        threading.Thread(target=turn_worker, daemon=True).start()
        return

    # FORWARD/STOP = continuous state command.
    with _command_lock:
        if _motor_request_in_flight:
            return

        if (
            direction == _last_sent_dir
            and (now - _last_sent_time) < SAME_COMMAND_HEARTBEAT
        ):
            return

        _motor_request_in_flight = True
        _last_sent_dir = direction
        _last_sent_time = now

    def state_worker():
        global _motor_request_in_flight
        try:
            _http_motor(direction)
        finally:
            with _command_lock:
                _motor_request_in_flight = False

    threading.Thread(target=state_worker, daemon=True).start()


In [6]:
# ==================== FACE DETECTION HELPERS ====================
def detection_to_box(detection, frame_w, frame_h):
    """Convert MediaPipe's normalized bounding box to (x, y, w, h) pixels."""
    bbox = detection.location_data.relative_bounding_box

    x = int(bbox.xmin * frame_w)
    y = int(bbox.ymin * frame_h)
    w = int(bbox.width * frame_w)
    h = int(bbox.height * frame_h)

    # Clamp the box to the actual frame. MediaPipe boxes can extend slightly outside it.
    x = max(0, min(x, frame_w - 1))
    y = max(0, min(y, frame_h - 1))
    w = max(1, min(w, frame_w - x))
    h = max(1, min(h, frame_h - y))

    return (x, y, w, h)


def detection_score(detection):
    """Return MediaPipe's first/primary detection confidence."""
    scores = detection.score
    return float(scores[0]) if scores else 0.0


def choose_largest_face(detections, frame_w, frame_h):
    """Choose the largest detected face, matching the original notebook's behavior."""
    candidates = []
    for detection in detections:
        box = detection_to_box(detection, frame_w, frame_h)
        x, y, w, h = box
        area = w * h
        candidates.append((area, detection_score(detection), box, detection))

    if not candidates:
        return None

    # Largest area first; confidence breaks ties.
    return max(candidates, key=lambda item: (item[0], item[1]))


def smooth_box(previous_box, new_box, alpha=FACE_SMOOTHING_ALPHA):
    """Exponential moving average for a steadier face box."""
    if previous_box is None:
        return new_box

    old = np.array(previous_box, dtype=np.float32)
    new = np.array(new_box, dtype=np.float32)
    smoothed = (1.0 - alpha) * old + alpha * new
    return tuple(int(v) for v in smoothed)


In [7]:
# ==================== DIRECTION LOGIC ====================
def decide_direction(face_box, frame_w, frame_h):
    """face_box = (x, y, w, h) in pixels.
    Returns one of: forward, left, right, stop.

    This preserves the original notebook's rules:
      - large face -> stop
      - face left of deadzone -> left
      - face right of deadzone -> right
      - otherwise -> forward
    """
    x, y, w, h = face_box
    face_center_x = x + w / 2.0
    frame_center_x = frame_w / 2.0
    offset = (face_center_x - frame_center_x) / frame_w

    face_width_ratio = w / frame_w

    if face_width_ratio > CLOSE_FACE_RATIO:
        return "stop"

    if offset < -CENTER_DEADZONE:
        return "left"
    elif offset > CENTER_DEADZONE:
        return "right"
    else:
        return "forward"


def stabilize_direction(candidate):
    """Require a candidate command to persist for a few frames before switching."""
    global _pending_direction, _pending_count, _stable_direction

    if candidate == _pending_direction:
        _pending_count += 1
    else:
        _pending_direction = candidate
        _pending_count = 1

    if _pending_count >= DIRECTION_CONFIRM_FRAMES:
        _stable_direction = candidate

    return _stable_direction


_pending_direction = None
_pending_count = 0
_stable_direction = "stop"


In [8]:
# ==================== MAIN FACE FOLLOWER LOOP ====================
def run_face_follower(show_window=True):
    """Read the ESP32-CAM MJPEG stream, detect the selected face,
    decide a direction, and send motor commands to the ESP32.
    Press Q to stop.
    """
    global _last_sent_dir, _last_sent_time
    global _pending_direction, _pending_count, _stable_direction

    cap = cv2.VideoCapture(STREAM_URL)

    if not cap.isOpened():
        print("ERROR: Cannot open ESP32-CAM stream.")
        print("Check ESP32_IP and open this URL in a browser first:")
        print(STREAM_URL)
        return

    # Keep the old command state clean when starting a new run.
    _last_sent_dir = None
    _last_sent_time = 0.0
    _pending_direction = None
    _pending_count = 0
    _stable_direction = "stop"

    previous_box = None
    no_face_count = 0

    print("Face follower started.")
    print("Stream:", STREAM_URL)
    print("Press Q to stop.")

    try:
        while True:
            ok, frame = cap.read()

            if not ok or frame is None:
                print("[video] failed to read frame")
                time.sleep(0.05)
                continue

            frame_h, frame_w = frame.shape[:2]

            # MediaPipe requires RGB input.
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_detector.process(rgb)

            selected = None

            if results.detections:
                selected = choose_largest_face(
                    results.detections,
                    frame_w,
                    frame_h
                )

            if selected is not None:
                _, confidence, box, _ = selected

                # Smooth the selected face position.
                previous_box = smooth_box(previous_box, box)

                direction_candidate = decide_direction(
                    previous_box,
                    frame_w,
                    frame_h
                )
                direction = stabilize_direction(direction_candidate)

                no_face_count = 0

                x, y, w, h = previous_box

                # Draw face box.
                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Face {confidence:.2f}",
                    (x, max(25, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

                # Draw frame center and face center.
                frame_cx = frame_w // 2
                face_cx = x + w // 2

                cv2.line(
                    frame,
                    (frame_cx, 0),
                    (frame_cx, frame_h),
                    (255, 255, 0),
                    1
                )

                cv2.circle(
                    frame,
                    (face_cx, y + h // 2),
                    5,
                    (0, 0, 255),
                    -1
                )

                # Send the stabilized motor command.
                send_direction(direction)

                cv2.putText(
                    frame,
                    f"COMMAND: {direction.upper()}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 255),
                    2
                )

            else:
                no_face_count += 1

                # Do not immediately stop for one bad frame.
                if no_face_count >= NO_FACE_STOP_FRAMES:
                    previous_box = None
                    direction = stabilize_direction("stop")
                    send_direction("stop")

                cv2.putText(
                    frame,
                    "NO FACE",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

            if show_window:
                cv2.imshow(WINDOW_NAME, frame)

                # Press Q to stop.
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

    except KeyboardInterrupt:
        print("Stopped by user.")

    finally:
        # Always stop the robot when the program exits.
        send_direction("stop")
        time.sleep(0.15)

        cap.release()

        if show_window:
            cv2.destroyAllWindows()

        print("Face follower stopped safely.")


In [9]:
# ==================== RUN ====================
# Start the face follower.
# Press Q in the OpenCV window to stop.
run_face_follower(show_window=True)


Face follower started.
Stream: http://10.158.116.178:81/stream
Press Q to stop.
[motor] stop failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=stop (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x280ff162ed0>, 'Connection to 10.158.116.178 timed out. (connect timeout=3.0)'))
[motor] stop -> 200
[motor] forward -> 200
[motor] forward -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] forward -> 200
[motor] forward -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] forward -> 200
[motor] forward -> 200
[motor] stop -> 200
[motor] stop -> 200
[motor] left -> 200
[motor] stop -> 200
[motor] left -> 200
[motor] stop -> 200
[motor] left -> 200
[motor] stop -> 200
[motor] forward -> 200
[motor] forward -> 200
[motor] forw

[motor] stop failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=stop (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x280ff14d350>, 'Connection to 10.158.116.178 timed out. (connect timeout=3.0)'))


## V3 tuning notes

Start with the supplied values before changing anything.

- If RIGHT still overshoots: lower `RIGHT_TURN_PULSE_MS` from 65 → 55 → 45 ms.
- If RIGHT becomes too weak: raise it in 10 ms steps.
- If LEFT begins overshooting: reduce `LEFT_TURN_PULSE_MS`.
- If the robot is centered but still does not move forward reliably, that is no longer a steering-software problem. FORWARD is already commanded continuously at the maximum available ON/OFF drive with ENA/ENB tied high, so the next checks are motor startup torque, L298N voltage drop/current capability, wheel friction, and the boost converter under startup load.
